# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pr120107/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules

REPO_URL = "https://github.com/pr120107/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        print("Cloning repository...")
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
            check=True
        )

    os.chdir(REPO_DIR)

    print("Installing requirements...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
        check=True
    )

print("Working directory:", os.getcwd())

Cloning repository...
Installing requirements...
Working directory: /content/flyrank-ml-internship


###1. Task Type: Clustering

My lane is an unsupervised clustering task because the goal is to discover natural groups of content pages based on their characteristics and performance patterns rather than predict a predefined label. The discovered content archetypes help content reviewers understand the different types of pages in their inventory and support decisions such as protecting, refreshing, improving, monitoring, merging, or pruning content.

###2. Target or Proxy

This is an unsupervised clustering task, so there is no target label to predict. Instead of learning from predefined labels, the model identifies natural groups of content pages based on similarities in their characteristics and performance metrics.

The discovered clusters become content archetypes that help reviewers understand different types of pages and support content management decisions. The cluster names are assigned by a human analyst after examining the characteristics of each group.

In [2]:
import pandas as pd

# Load the dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Features that could be used for clustering
features = [
    "clicks_90d",
    "impressions_90d",
    "ctr",
    "engagement_rate",
    "content_age_days"
]

# Display a few rows of these features
df[features].head()

,clicks_90d,impressions_90d,ctr,engagement_rate,content_age_days
0,29,3803,0.76,5.88,187
1,7,15320,0.05,0.00,445
2,11,12581,0.09,0.00,141
3,58,11751,0.49,1.28,463
4,24,19140,0.13,0.00,263


###3. Success Metric

I will use the **Silhouette Score** as the primary evaluation metric because it measures how well-separated and cohesive the discovered clusters are. A higher Silhouette Score indicates that pages within the same cluster are more similar to each other than to pages in other clusters.

Since clustering is intended to support content management decisions, I will also perform a human sense-check by examining whether the discovered archetypes are meaningful, interpretable, and useful for content reviewers.

In [3]:
from sklearn.metrics import silhouette_score

print("Chosen evaluation metric for clustering: Silhouette Score")
print("The Silhouette Score will be calculated after clustering is performed.")

Chosen evaluation metric for clustering: Silhouette Score
The Silhouette Score will be calculated after clustering is performed.


###4. Unit of Analysis

One row represents one content page in the starter dataset. Each row contains performance metrics and content characteristics for a single page over the trailing 90-day period.

These pages are the units that will be compared during clustering to discover natural content archetypes. The resulting clusters can then support content management decisions such as refreshing, protecting, monitoring, or pruning groups of similar pages.

In [4]:
import pandas as pd

# Load the dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Display basic information
print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst five rows:")
df.head()

Dataset shape: (30000, 44)

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

First five rows:


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


###5. Why ML Beats a Fixed Rule

A fixed set of if-statements would be difficult to maintain because content pages are described by many interacting features such as CTR, impressions, engagement, sessions, search position, and content age. The relationships between these features are too complex to capture with a small number of hand-written rules.

Clustering can discover natural patterns across many features simultaneously, allowing similar pages to be grouped into meaningful content archetypes. These archetypes provide decision support for content reviewers without requiring manually defined rules for every possible combination.

In [5]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(f"Number of content pages: {len(df):,}")
print(f"Number of available features: {df.shape[1]}")

print("\nThe large number of pages and features suggests that manually writing rules for every possible combination would not scale well.")

Number of content pages: 30,000
Number of available features: 44

The large number of pages and features suggests that manually writing rules for every possible combination would not scale well.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.